# Extracting scGPT finetuned model attention weights

This notebook was modified from [scGPT_finetune](https://github.com/meconsens/genome-head-interpreter/blob/main/attentions/scGPT/scGPT_finetune.ipynb) & [Tutorial_Attention_GRN](https://github.com/bowang-lab/scGPT/blob/main/tutorials/Tutorial_Attention_GRN.ipynb). Purpose to extract attention weights from model which has been trained to specifically output attention weights used. 

## 0. Imports, Variables

In [ ]:
# imports
import copy
import json
import os
from pathlib import Path
import sys
import warnings

import torch
from anndata import AnnData
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import pandas as pd
import tqdm

from scipy.sparse import issparse
import scipy as sp
from torch.nn.functional import softmax
from tqdm import tqdm
import pandas as pd

from torchtext.vocab import Vocab
from torchtext.vocab import (
    Vocab as VocabPybind,
)

sys.path.insert(0, "../")

import scgpt as scg
from scgpt.tasks import GeneEmbedding
from scgpt.tokenizer.gene_tokenizer import GeneVocab
from scgpt.model import TransformerModel
from scgpt.utils import set_seed 
from scgpt.tokenizer import tokenize_and_pad_batch
from scgpt.preprocess import Preprocessor

os.environ["KMP_WARNINGS"] = "off"
warnings.filterwarnings('ignore')

# variables
set_seed(42)
pad_token = "<pad>"
special_tokens = [pad_token, "<cls>", "<eoc>"]
n_hvg = 1200
n_bins = 51
mask_value = -1
pad_value = -2
n_input_bins = n_bins

path_model = "../outputs/run-25-07-10-01/"




## 1. Load Model & Data


In [ ]:
# Specify model path; here we load the scGPT blood model fine-tuned on adamson
# model_dir = Path("../save/finetuned_scGPT_adamson")
model_config_file = os.path.join(path_model, "args.json")
model_file = os.path.join(path_model, "model_1.pt")
vocab_file = os.path.join(path_model, "vocab.json")

vocab = GeneVocab.from_file(vocab_file)
for s in special_tokens:
    if s not in vocab:
        vocab.append_token(s)

# Retrieve model parameters from config files
with open(model_config_file, "r") as f:
    model_configs = json.load(f)
print(
    f"Resume model from {model_file}, the model args will override the "
    f"config {model_config_file}."
)
embsize = model_configs["embsize"]
nhead = model_configs["nheads"]
d_hid = model_configs["d_hid"]
nlayers = model_configs["nlayers"]
# n_layers_cls = model_configs["n_layers_cls"]

gene2idx = vocab.get_stoi()


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ntokens = len(vocab)  # size of vocabulary
model = TransformerModel(
    ntokens,
    embsize,
    nhead,
    d_hid,
    nlayers,
    vocab=vocab,
    pad_value=pad_value,
    n_input_bins=n_input_bins,
    use_fast_transformer=True,
    domain_spec_batchnorm = "batchnorm",    #! I added this otherwise self.bn will not initialize
                                            #! https://github.com/bowang-lab/scGPT/issues/134
)

try:
    model = (torch.load(model_file))
    print(f"Loading all model params from {model_file}")
except:
    # only load params that are in the model and match the size
    model_dict = model.state_dict()
    pretrained_dict = torch.load(model_file)
    pretrained_dict = {
        k: v
        for k, v in pretrained_dict.items()
        if k in model_dict and v.shape == model_dict[k].shape
    }
    for k, v in pretrained_dict.items():
        print(f"Loading params {k} with shape {v.shape}")
        model_dict.update(pretrained_dict)
        model.load_state_dict(model_dict)

model.to(device)


# Load training data
adata_file = os.path.join(path_model, "adata_train_1.h5ad")
adata = sc.read(adata_file)
data_is_raw = False
print(adata.X.shape)

## 2. Retrieve scGPT's attention weights

In [ ]:
model.eval()
dict_attn_by_signature = {}

with torch.no_grad():
    for batch_num, batch_data in enumerate(test_loader):
        input_ids = batch_data["gene_ids"].to(device)
        input_values = batch_data["values"].to(device)
        src_key_padding_mask = input_ids.eq(vocab[pad_token])

        # Forward pass with output_attentions=True
        output_dict = model(
            input_ids,
            input_values,
            src_key_padding_mask=src_key_padding_mask,
            CLS=True
        )

        # Get all attentions: List[num_layers] of [B, num_heads, L, L]
        all_attn = output_dict["attentions"]

        # Choose specific layer and head (example: layer 0, head 0)
        layer = 0
        head = 0
        attn_matrices = all_attn[layer][:, head, :, :]  # Shape: [B, L, L]

        for i, attn_matrix in enumerate(attn_matrices):  # One per sample
            # Take max over rows → how much attention each token **receives**
            max_received = attn_matrix.max(dim=0)[0].detach().cpu().numpy()
            dict_attn_by_signature[batch_num * batch_size + i] = max_received


In [ ]:
def analyze_attention_heads(model, adata, device, vocab, nlayers, nhead, pad_token, config):
    """
    Analyze attention heads directly from AnnData object
    
    Args:
        model: The model to analyze
        adata: AnnData object containing data
        device: Torch device
        vocab: Vocabulary
        nlayers: Number of layers in model
        nhead: Number of attention heads per layer
        pad_token: Token to use for padding
        config: Configuration object
        
    Returns:
        Dictionary with attention scores for each layer and head
    """
    # Process data for testing
    all_counts = (
        adata.layers[input_layer_key].A
        if issparse(adata.layers[input_layer_key])
        else adata.layers[input_layer_key]
    )

    celltypes_labels = adata.obs["celltype_id"].tolist()  # make sure count from 0
    celltypes_labels = np.array(celltypes_labels[:499])

    batch_ids = adata.obs["batch_id"].tolist()
    batch_ids = np.array(batch_ids[:499])

    tokenized_test = tokenize_and_pad_batch(
        all_counts[:499],
        gene_ids,
        max_len=max_seq_len,
        vocab=vocab,
        pad_token=pad_token,
        pad_value=pad_value,
        append_cls=True,  # append <cls> token at the beginning
        include_zero_gene=include_zero_gene,
    )

    input_values_test = random_mask_value(
        tokenized_test["values"],
        mask_ratio=mask_ratio,
        mask_value=mask_value,
        pad_value=pad_value,
    )

    test_data_pt = {
        "gene_ids": tokenized_test["genes"],
        "values": input_values_test,
        "target_values": tokenized_test["values"],
        "batch_labels": torch.from_numpy(batch_ids).long(),
        "celltype_labels": torch.from_numpy(celltypes_labels).long(),
    }

    test_loader = DataLoader(
        dataset=SeqDataset(test_data_pt),
        batch_size=eval_batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=min(len(os.sched_getaffinity(0)), eval_batch_size // 2),
        pin_memory=True,
    )

    # settings for prediction
    MLM = False  # whether to use masked language modeling, currently it is always on.
    CLS = True  # celltype classification objective
    ADV = False  # Adversarial training for batch correction
    CCE = False  # Contrastive cell embedding objective
    MVC = config.MVC  # Masked value prediction for cell embedding
    ECS = config.ecs_thres > 0  # Elastic cell similarity objective
    DAB = False  # Domain adaptation by reverse backpropagation, set to 2 for separate optimizer
    INPUT_BATCH_LABELS = False  # TODO: have these help MLM and MVC, while not to classifier
    input_emb_style = "continuous"  # "category" or "continuous" or "scaling"
    cell_emb_style = "cls"  # "avg-pool" or "w-pool" or "cls"
    adv_E_delay_epochs = 0  # delay adversarial training on encoder for a few epochs
    adv_D_delay_epochs = 0
    mvc_decoder_style = "inner product"
    ecs_threshold = config.ecs_thres
    dab_weight = config.dab_weight

    explicit_zero_prob = MLM and include_zero_gene  # whether explicit bernoulli for zeros
    do_sample_in_train = False and explicit_zero_prob  # sample the bernoulli in training

    num_layers = nlayers 
    num_heads = nhead  

    model.to(device)
    model.eval()

    # Log analysis info
    logger.info("\n====== Analyzing Attention Heads ======")
    logger.info(f"Model has {num_layers} layers with {num_heads} heads each")
    
    # Print class distribution in dataset
    logger.info("\n====== Class Distribution in Dataset ======")
    test_labels = []
    for batch in test_loader:
        test_labels.extend(batch["celltype_labels"].numpy())

    unique_test_labels, test_counts = np.unique(test_labels, return_counts=True)
    test_percentages = test_counts / len(test_labels) * 100

    logger.info("Dataset class distribution:")
    for label, count, percentage in zip(unique_test_labels, test_counts, test_percentages):
        logger.info(f"  Class {label} ({id2type[label]}): {count} samples ({percentage:.2f}%)")

    # Dictionary to store all examples and their scores for each head
    examples_scores_attention = {layer: {head: [] for head in range(num_heads)} for layer in range(num_layers)}

    for batch_num, batch_data in enumerate(test_loader):
        input_gene_ids = batch_data["gene_ids"].to(device)
        input_values = batch_data["values"].to(device)
        target_values = batch_data["target_values"].to(device)
        batch_labels = batch_data["batch_labels"].to(device)
        celltype_labels = batch_data["celltype_labels"].to(device)
        src_key_padding_mask = input_gene_ids.eq(vocab[pad_token])

        # Output dictionary 
        output_dict = model(
            input_gene_ids,
            input_values,
            src_key_padding_mask=src_key_padding_mask,
            batch_labels=batch_labels if INPUT_BATCH_LABELS or config.DSBN else None,
            CLS=CLS,
            CCE=False,
            MVC=False,
            ECS=False,
            do_sample=do_sample_in_train,
        )
        
        input_tokens = [vocab.lookup_tokens(ids.tolist()) for ids in input_gene_ids]
        outputs = output_dict["cls_output"]
       
        # Get attention scores
        all_attentions = output_dict["attentions"]  # assuming the model was set with output_attentions=True
        # Print shape of attention outputs
        print(f"all_attentions contains {len(all_attentions)} layers")
        
        # Get labels
        batch_labels_list = celltype_labels.detach().cpu().numpy().tolist()
        
        # Get expression values
        input_values_list = input_values.detach().cpu().numpy().tolist()
        
        if batch_num == 0:
            logger.info(f"Processing {len(input_values_list)} examples per batch")

        # For each layer and head...
        for layer in range(num_layers):
            # Check shape of attention for this layer
            print(f"Layer {layer} attention shape: {all_attentions[layer].shape}")
            
            for head in range(num_heads):
                attention_scores = all_attentions[layer][:, head, :, :]
                # check shape per head
                print(f"Layer {layer}, Head {head}, Attention shape: {attention_scores.shape}")

                # Tokens, attention matrices, labels, and input_values together
                for i, (tokens, att_matrix, label, values) in enumerate(zip(input_tokens, attention_scores, batch_labels_list, input_values_list)):
                    # Check indiividual attention matrix shape
                    if i == 0:  # Just print for the first item to avoid flooding output
                        print(f"Sample 0, Layer {layer}, Head {head}, Matrix shape: {att_matrix.shape}")
                        print(f"Min: {att_matrix.min().item()}, Max: {att_matrix.max().item()}, Mean: {att_matrix.mean().item()}")
                        
                    max_att_scores = att_matrix.max(dim=0)[0].detach().cpu().numpy()
                    # Append a tuple with max attention scores, tokens, label, and the specific input_values
                    examples_scores_attention[layer][head].append((max_att_scores, tokens, label, values))
    
    # Log completion
    total_examples = len(examples_scores_attention[0][0])
    logger.info(f"\nAttention analysis complete! Processed {total_examples} total examples")
    
    return examples_scores_attention

In [ ]:
results = analyze_attention_heads(model, 
                                  adata_test, 
                                  device, 
                                  vocab, 
                                  nlayers,
                                  nhead, 
                                  pad_token, 
                                  config)

In [ ]:
# unpack results
examples_scores_attention  = results

# directory exists
os.makedirs(f'{full_path}/attention/{dataset_name}/{task_name}/', exist_ok=True)
#save scores in layer-indexed-files
for layer in range(nlayers):
    #examples_scores_attention for the layer
    attention_filename = f'{full_path}/attention/{dataset_name}/{task_name}/examples_scores_attention_layer{layer}.p'
    with open(attention_filename, 'wb') as f:
        pickle.dump(examples_scores_attention[layer], f)
    print(f'Attentions saved to {attention_filename}')

### Attention Heatmap

In [ ]:
# Define the path to the directory containing the pickle files
base = "" # path to attentions directory
data_dir = os.path.join(base, dataset, task_name)

pkl_files = [
    'examples_scores_attention_layer0.p',
    'examples_scores_attention_layer1.p',
    'examples_scores_attention_layer2.p',
    'examples_scores_attention_layer3.p',
    'examples_scores_attention_layer4.p',
    'examples_scores_attention_layer5.p',
    'examples_scores_attention_layer6.p',
    'examples_scores_attention_layer7.p',
    'examples_scores_attention_layer8.p',
    'examples_scores_attention_layer9.p',
    'examples_scores_attention_layer10.p',
    'examples_scores_attention_layer11.p',
]

In [ ]:
# Load all pickle files into memory
layers_data = []
for file_name in pkl_files:
    file_path = os.path.join(data_dir, file_name)
    with open(file_path, 'rb') as f:
        layers_data.append(pickle.load(f))

# Assuming all layers have the same number of heads and all heads have the same number of cells
num_layers = len(layers_data)
num_heads = len(layers_data[0])
num_cells = len(layers_data[0][0])
num_genes = len(layers_data[0][0][0][0]) - 1  # Assuming each cell contains data for the same number of genes

print('Check Data:', num_layers, num_heads, num_cells, num_genes)

In [ ]:
mean_score_df:pd.DataFrame = pd.DataFrame()

# Open the layer
for layer in range(12):
    # print("LAYER:", layer)
    with open(f'{data_dir}/examples_scores_attention_layer{layer}.p', 'rb') as f:
        results:dict = pickle.load(f)

    # Calculate mean score per sequence per head
    tmp_dict:dict = {}
    for head in results:
        # print("HEAD:", head)
        tmp_list:list = []
        for i in range(len(results[head])):
          # sequence = (results[head][i][0]).cpu().numpy()
          # modified_sequence = sequence[1:-1]  # Exclude the first and last token
          # tmp_list.append(np.mean(modified_sequence))
          tmp_list.append(np.mean((results[head][i][0])))
        tmp_dict[head] = tmp_list

    # Merge layer scores
    tmp_df:pd.DataFrame = pd.DataFrame(tmp_dict)
    tmp_df.columns = [f'head{i}' for i in range(8)]
    tmp_df['layer'] = f'layer{layer}'
    mean_score_df = pd.concat([mean_score_df, tmp_df])

# Save the results as a CSV
mean_score_df.to_csv(f'{data_dir}/examples_mean_{approach}_scores.csv', index=False)

del(tmp_df, tmp_dict, tmp_list, results, head, layer, f, i)

In [ ]:
# Calculate mean score for each layer
tmp_mean_df = mean_score_df.groupby('layer').mean()

# Normalize the scores within each layer
tmp_mean_df = tmp_mean_df.apply(lambda x: (x - x.min()) / (x.max() - x.min()), axis=1)

# Sort layers
tmp_mean_df = tmp_mean_df.reindex(sorted(tmp_mean_df.index, key=lambda x: int(x[5:])))

# Plotting
plt.figure(1, figsize=(9, 6))
sns.set(color_codes=True)
sns.set(font_scale=0.9)
ax = sns.heatmap(tmp_mean_df, cmap='GnBu', cbar_kws={'label': 'Scale'})
ax.set(ylabel="Layers", xlabel="Heads")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, horizontalalignment='right')
ax.set_yticklabels(ax.get_yticklabels(), rotation=45)

plt.savefig(f"{data_dir}/scgpt_mean_attention_heatmap.png", dpi=300, bbox_inches='tight')

plt.show()

tmp_mean_df.to_csv(f'{data_dir}/tmp_mean_{approach}_scores.csv', index=False)

# Clean up
del(tmp_mean_df, ax)